# Vaani Track 1 — TRACE full run

**T**ranscript-**R**egularised, count-exact **E**vent decoding on a fine-tuned ATST-Frame + BEATs model.

What this notebook does, end to end, in one T4 x2 session:

1. trains two `TraceModel`s (different seeds / optimisers) on corpus gold + silver, **all five
   validation folds**, and the synthetic re-mixer built from validation's clean/noisy pairs;
2. runs full-length inference on the test audio (8 s windows, 4 s hop — nothing truncated);
3. decodes each clip with **exactly K events**, K = the transcript's noise-tag count when the
   test metadata carries transcripts, otherwise the count head;
4. natural clips: the v2 span model's proposals re-ranked by TraceModel posteriors;
   synthetic clips: TraceModel presence decode;
5. writes `/kaggle/working/submission.zip` (`predictions.jsonl` at the root).

### Measured on the held-out validation fold (fold 0, 496 clips; trained on the other 4)

| decoder | NAT | SYN | ALL |
|---|---|---|---|
| v2 span model, count head (old pipeline) | 1.133 | 1.634 | 1.330 |
| v2 span model, K from transcript | 1.177 | 1.841 | 1.428 |
| **TRACE: span proposals re-ranked by TraceModel + TraceModel SYN** | **1.197** | **1.983\*** | **1.498** |

\* the synthetic clips reuse a library of ~117 noise recordings, so the held-out fold shares
noise sources with training; the leakage-free synthetic floor is 1.84. The test's synthetic
third almost surely reuses the same library.

### Before you run
* **Accelerator:** GPU T4 x2. **Internet:** on.
* **Add Data** (all private):
  1. the packed corpus — output of the `vaani-data` notebook (`vaani/manifest.jsonl` + `audio_*.bin`);
  2. the validation set (`validation/validationMetadata.json` + audio folders);
  3. the **test** set from Codabench (`input_data`) uploaded as a private dataset;
  4. the v2 span-model checkpoint `runs/f0/best.pt` (output of `vaani-train-f0`; on another
     account, upload that one file as a private dataset).
* Never add the Codabench `reference_data` package.

In [ ]:
# ============================== CONFIG ==============================
BRANCH      = "trace"
STEPS       = 3000        # per model; held-out natural peaked at step 3000 (E3: 1.151), fell by 3500
HOURS_CAP   = 4.5         # per model, safety stop
QUOTAS      = '{"gold":0.45,"silver":0.10,"valnat":0.10,"remix":0.25,"valsyn":0.10}'
MODELS      = [("adamw", 0), ("muon", 1)]    # (optimiser, seed) per ensemble member
SYN_DECODE  = {"decoder": "thr", "kw": {"thr": 0.7, "med": 1, "min_dur": 0.1}}
USE_TRANSCRIPTS = True    # K from the test transcripts when present
# =====================================================================

In [ ]:
import os, sys, glob, shutil, subprocess, json, time
SRC = "/tmp/v2"
shutil.rmtree(SRC, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                "https://github.com/raut7218/vaani-sed-v2.git", SRC], check=True)
os.chdir(SRC); sys.path.insert(0, SRC)
subprocess.run("pip -q install -r requirements.txt 2>&1 | tail -1", shell=True)
subprocess.run("pip -q install uroman 2>&1 | tail -1", shell=True)
subprocess.run("python scripts/fetch_encoders.py --all 2>&1 | tail -2", shell=True)
import torch
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "set Accelerator to GPU T4 x2"

In [ ]:
def one(pattern, what, required=True):
    hits = [p for p in glob.glob(pattern, recursive=True) if "__MACOSX" not in p]
    if not hits and required:
        raise FileNotFoundError(f"{what} not found ({pattern}) - add it under Add Data")
    return hits[0] if hits else ""

VAL_META = one("/kaggle/input/**/validationMetadata.json", "validation metadata")
CORPUS   = os.path.dirname(one("/kaggle/input/**/vaani/manifest.jsonl", "packed corpus manifest"))
F0_CKPT  = one("/kaggle/input/**/runs/f0/best.pt", "v2 span-model checkpoint", required=False) \
           or one("/kaggle/input/**/best.pt", "v2 span-model checkpoint", required=False)
# the test set is whichever input holds wav files that are not the validation set
val_root = os.path.dirname(VAL_META)
test_wavs = [p for p in glob.glob("/kaggle/input/**/*.wav", recursive=True)
             if not p.startswith(val_root) and "__MACOSX" not in p]
TEST_DIR = os.path.commonpath(test_wavs) if test_wavs else ""
print("validation :", VAL_META)
print("corpus     :", CORPUS)
print("span model :", F0_CKPT or "MISSING - natural clips fall back to TraceModel decoding")
print("test audio :", TEST_DIR, f"({len(test_wavs)} wav files)")
assert TEST_DIR, "test audio not found - add the Codabench input_data as a dataset"

## Train

Each member is a DDP run on both T4s. `--hold-fold -1` trains on every validation fold (no
held-out evaluation inside this notebook — that was done in the experiments). A checkpoint is
written every 1000 steps, so a session that dies keeps its last one.

In [ ]:
CKPTS = []
for opt, seed in MODELS:
    out = f"/kaggle/working/trace_{opt}_s{seed}"
    cmd = (f"torchrun --nproc_per_node 2 -m tracesed.train --val-meta '{VAL_META}' --corpus '{CORPUS}' "
           f"--steps {STEPS} --max-steps-time {HOURS_CAP} --opt {opt} --seed {seed} --hold-fold -1 "
           f"--eval-every 1000 --quotas '{QUOTAS}' --bs 12 --workers 2 --out {out}")
    print(cmd, flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, env={**os.environ, "PYTHONPATH": SRC, "OMP_NUM_THREADS": "1"})
    print(f"exit {r.returncode} after {(time.time()-t0)/60:.0f} min", flush=True)
    if os.path.exists(f"{out}/model.pt"):
        CKPTS.append(f"{out}/model.pt")
print("checkpoints:", CKPTS)
assert CKPTS, "no model trained"

## Predict

Natural clips: exactly K of the span model's proposals, re-ranked by TraceModel presence, edge
contrast and boundary evidence (`tracesed/fuse.py`). Synthetic clips: TraceModel presence
decode. K = transcript noise-tag count when available, else the count head.

In [ ]:
json.dump({"natural": {"decoder": "thr", "kw": {"thr": 0.5, "med": 5, "min_dur": 0.1}}, "synthetic": SYN_DECODE},
          open("/kaggle/working/decode.json", "w"))
cmd = (f"python -m tracesed.predict --ckpt {' '.join(CKPTS)} --audio-dir '{TEST_DIR}' "
       f"--decode /kaggle/working/decode.json --out /kaggle/working/submission.zip"
       + (f" --f0-ckpt '{F0_CKPT}'" if F0_CKPT else "")
       + ("" if USE_TRANSCRIPTS else " --no-transcript"))
print(cmd, flush=True)
r = subprocess.run(cmd, shell=True, env={**os.environ, "PYTHONPATH": SRC})
assert r.returncode == 0

In [ ]:
import zipfile, numpy as np
with zipfile.ZipFile("/kaggle/working/submission.zip") as z:
    assert z.namelist() == ["predictions.jsonl"], z.namelist()
    rows = [json.loads(l) for l in z.read("predictions.jsonl").decode().splitlines() if l.strip()]
ids = [r["clip_id"] for r in rows]
assert len(ids) == len(set(ids)), "duplicate clip_id"
bad = [r for r in rows for e in r["events"] if not (0 <= e["onset"] < e["offset"])]
assert not bad, bad[:3]
n = np.array([len(r["events"]) for r in rows])
print(f"{len(rows)} clips | events/clip {n.mean():.2f} (validation: natural 1.63, synthetic 3.18) | empty {100*(n==0).mean():.1f}%")